# 1. Khởi tạo môi trường & Đọc dữ liệu

In [1]:

# export
import re
import pandas as pd
from helpers.get_configs import get_config

RAW_DATA_PATH = get_config('dirs.yaml', ['raw_data', 'score_blocks'], as_path=True)
CLEANED_DATA_PATH = get_config('dirs.yaml', ['cleaned_data', 'score_blocks'], as_path=True)
SCRIPT_OUTPUT_PATH = get_config('dirs.yaml', ['unit_scripts', 'score_blocks'], as_path=True)

# Đọc file CSV vào DataFrame
df = pd.read_csv(RAW_DATA_PATH, encoding='utf-8')


In [2]:
# Khảo sát thông tin tổng quan của dữ liệu
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 280 entries, 0 to 279
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   code      280 non-null    str  
 1   subject1  280 non-null    str  
 2   subject2  280 non-null    str  
 3   subject3  280 non-null    str  
 4   subject4  1 non-null      str  
dtypes: str(5)
memory usage: 21.8 KB


,code,subject1,subject2,subject3,subject4
0,D01,Toán,Ngữ văn,Tiếng anh,NaN
1,A00,Toán,Vật lí,Hóa học,NaN
2,A01,Toán,Vật lí,Tiếng anh,NaN
3,D07,Toán,Hóa học,Tiếng anh,NaN
4,B00,Toán,Hóa học,Sinh học,NaN


# 2. Kiểm tra giá trị rỗng

In [3]:
# Kiểm tra số lượng giá trị rỗng trên các cột
df.isna().sum()

code          0
subject1      0
subject2      0
subject3      0
subject4    279
dtype: int64

**Kết luận**: Các cột bị thiếu dữ liệu đều thuộc về `subject4` (một tổ hợp môn có thể có 4 môn học hoặc 3 môn học), dữ liệu hợp lý, nên không cần bổ sung / lược bỏ các hàng này.

# 3. Chuẩn hóa dữ liệu

In [4]:
# export
def normalize_text(text):
    if pd.isna(text):
        return text
    
    # 1. Loại bỏ khoảng trắng đầu/cuối
    text = text.strip()
    
    # 2. Loại bỏ 2 dấu cách liên tiếp trở lên -> thành 1 dấu cách
    text = re.sub(r'\s+', ' ', text)
    
    # 3. Đưa về chữ thường toàn bộ trước
    text = text.lower()
    
    # 4. Thay thế "lý" thành "lí"
    text = text.replace('lý', 'lí')
    
    # 5. In hoa chữ cái đầu tiên, còn lại giữ nguyên (chữ thường)
    if text:
        text = text[0].upper() + text[1:]
    
    return text

# Áp dụng cho 3 cột môn học
for col in ['subject1', 'subject2', 'subject3']:
    df[col] = df[col].apply(normalize_text)

In [5]:
# Khảo sát kiểm tra lại kết quả tần suất các môn
for col in ['subject1', 'subject2', 'subject3']:
    print(f"\n=== Cột: {col} ===")
    print(df[col].value_counts(dropna=False))


=== Cột: subject1 ===
subject1
Ngữ văn              136
Toán                 132
Ngữ văn               4
Văn                    3
Tư duy định lượng      3
Toán                  1
Vẽ mỹ thuật            1
Name: count, dtype: int64

=== Cột: subject2 ===
subject2
Vật lí                                     32
Lịch sử                                    31
Địa lí                                     28
Sinh học                                   27
Hóa học                                    25
Gdktpl                                     24
Tin học                                    14
Toán                                       13
Tiếng anh                                  13
Công nghệ nông nghiệp                      12
Công nghệ công nghiệp                      12
Ngữ văn                                     9
Năng khiếu 1                                3
Tư duy định tính                            3
Tiếng pháp                                  2
Tiếng nhật                                  2

# 4. Lưu dữ liệu đã làm sạch

In [6]:
# export
CLEANED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEANED_DATA_PATH, index=False)

In [7]:
# Khảo sát dữ liệu sau khi lưu
print(f"Tổng số dòng sau chuẩn hóa: {len(df)}")
df.info()
df.head()

Tổng số dòng sau chuẩn hóa: 280
<class 'pandas.DataFrame'>
RangeIndex: 280 entries, 0 to 279
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   code      280 non-null    str  
 1   subject1  280 non-null    str  
 2   subject2  280 non-null    str  
 3   subject3  280 non-null    str  
 4   subject4  1 non-null      str  
dtypes: str(5)
memory usage: 21.8 KB


,code,subject1,subject2,subject3,subject4
0,D01,Toán,Ngữ văn,Tiếng anh,NaN
1,A00,Toán,Vật lí,Hóa học,NaN
2,A01,Toán,Vật lí,Tiếng anh,NaN
3,D07,Toán,Hóa học,Tiếng anh,NaN
4,B00,Toán,Hóa học,Sinh học,NaN


# 5. Xuất Notebook sang Unit Script

In [8]:
from helpers.export_to_script import export_to_script

export_to_script(output_path=SCRIPT_OUTPUT_PATH)

Đã xuất 3 cell từ 'score-blocks.ipynb' sang 'D:\UNIDOCS\IS405.R11-Big-Data\scripts-processing\unit-scripts\score_blocks.py'


'D:\\UNIDOCS\\IS405.R11-Big-Data\\scripts-processing\\unit-scripts\\score_blocks.py'